# Implementing gradient descent for multiple regression

In the first notebook we explored multiple regression using Scikit-learn. Now we will use Pandas along with numpy to solve for the regression weights with gradient descent. 

In this notebook we will cover estimating multiple regression weights via gradient descent. We will:

- Add a constant column of 1's to a pandas DataFrame to account for the intercept
- Convert a pandas DataFrame into a numpy array
- Write a predict_output() function using numpy
- Write a numpy function to compute the derivative of the regression weights with respect to a single feature
- Write gradient descent function to compute the regression weights given an initial weight vector, step size and tolerance.
- Use the gradient descent function to estimate regression weights for multiple features 

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from sklearn.linear_model import LinearRegression

## Loading Data

In [2]:
data = pd.read_csv("../../data/kc_house_data.csv")
train_data = pd.read_csv("../../data/kc_house_train_data.csv")
test_data = pd.read_csv("../../data/kc_house_test_data.csv")

## Assisting Functions

Next write a function that takes:
- a data set,
- a list of features (e.g. [‘sqft_living’, ‘bedrooms’]), to be used as inputs,
- a name of the output (e.g. ‘price’).

This function should return:
- a features_matrix (2D array) consisting of first a column of ones followed by columns containing the values of the input features in the data set in the same order as the input list.
- an output_array which is an array of the values of the output in the data set (e.g. ‘price’).

In [12]:
def get_numpy_data(dataframe, features, output):
    # feature_vals is the actual numeric data
    features_vals = dataframe[features].to_numpy()
    features_matrix = np.column_stack( [np.ones(len(dataframe)), features_vals] )
    output_array = dataframe[output].to_numpy()
    return features_matrix, output_array

If the features matrix (including a column of 1s for the constant) is stored as a 2D array (or matrix) and the regression weights are stored as a 1D array then the predicted output is just the dot product between the features matrix and the weights (with the weights on the right). Write a function ‘predict_output’ which accepts a 2D array ‘feature_matrix’ and a 1D array ‘weights’ and returns a 1D array ‘predictions’.

In [4]:
def predict_outcome(feature_matrix, weights):
    return feature_matrix @ weights

If we have a the values of a single input feature in an array ‘feature’ and the prediction ‘errors’ (predictions - output) then the derivative of the regression cost function with respect to the weight of ‘feature’ is just twice the dot product between ‘feature’ and ‘errors’. Write a function that accepts a ‘feature’ array and ‘error’ array and returns the ‘derivative’ (a single number).

In [5]:
def feature_derivative(errors, feature):
    return 2 * (feature @ errors)

## Gradient Descent Algorithm

Now we will use our predict_output and feature_derivative to write a gradient descent function. Although we can compute the derivative for all the features simultaneously (the gradient) we will explicitly loop over the features individually for simplicity. Write a gradient descent function that does the following:
- Accepts a numpy feature_matrix 2D array, a 1D output array, an array of initial weights, a step size and a convergence tolerance.
- While not converged updates each feature weight by subtracting the step size times the derivative for that feature given the current weights
- At each step computes the magnitude/length of the gradient (square root of the sum of squared components)
- When the magnitude of the gradient is smaller than the input tolerance returns the final weight vector.

In [9]:
def gradient_descent_loop(feature_matrix, output_array, initial_weights, step_size, tolerance):
    converged = False
    weights = np.array(initial_weights)
    
    while not converged:
        # compute the predictions based on feature_matrix and weights
        predictions = predict_outcome(feature_matrix, weights)
        # compute the errors as predictions - output
        errors = predictions - output_array
        # initialize the gradient
        gradient_sum_squares = 0
        # while not converged, update each weight individually
        for i in range(len(weights)):
            # compute the derivative for weight[i]
            derivative = feature_derivative(errors, feature_matrix[:, i])
            # add the squared derivative to the gradient magnitude
            gradient_sum_squares += derivative ** 2
            # update the weight based on step_size and derivative
            weights[i] -= step_size * derivative
        gradient_magnitude = np.sqrt(gradient_sum_squares)
        if gradient_magnitude < tolerance:
            converged = True
    return weights

Here, I present the vectorized version; faster:

In [10]:
def gradient_descent_vectorized(feature_matrix, output_array, initial_weights, step_size, tolerance):
    converged = False
    weights = np.array(initial_weights)
    while not converged:
        predictions = feature_matrix @ weights
        errors = predictions - output_array
        gradient = 2 * (feature_matrix.T @ errors) # feature_matrix is NxD while errors is Nx1
        weights -= step_size * gradient
        gradient_magnitude = np.sqrt((gradient ** 2).sum())
        if gradient_magnitude < tolerance:
            converged = True
    return weights

## Using the Gradient Descent on Single Variable

Now we will run the regression gradient descent function on some actual data. In particular we will use the gradient descent to estimate the model from Week 1 using just an intercept and slope. Use the following parameters:
- features: ‘sqft_living’
- output: ‘price’
- initial weights: -47000, 1 (intercept, sqft_living respectively)
- step_size = 7e-12
- tolerance = 2.5e7

In [14]:
feature_matrix, output_array = get_numpy_data(train_data, ['sqft_living'], "price")

simple_weights = gradient_descent_vectorized(feature_matrix, output_array, [-47000.,1.], 7e-12, 2.5e7)

**Quiz Question**: What is the value of the weight for sqft_living -- the second element of ‘simple_weights’ (rounded to 1 decimal place)?

In [18]:
print ("Answer:", simple_weights[1].round(2))

Answer: 281.91


Now build a corresponding ‘test_simple_feature_matrix’ and ‘test_output’ using test_data. Using ‘test_simple_feature_matrix’ and ‘simple_weights’ compute the predicted house prices on all the test data.

In [20]:
test_simple_feature_matrix, test_output = get_numpy_data(test_data, ['sqft_living'], "price")
predicted_house_prices = test_simple_feature_matrix @ simple_weights

**Quiz Question**: What is the predicted price for the 1st house in the Test data set for model 1 (round to nearest dollar)?

In [28]:
first_house_price_simple_model = predicted_house_prices[0].round(0)
print ("Answer:", first_house_price_simple_model)

Answer: 356134.0


Now compute RSS on all test data for this model. Record the value and store it for later.

In [32]:
simple_RSS = ((predicted_house_prices - test_output) ** 2).sum()

## Using the Gradient Descent on Multiple Variables

Now we will use the gradient descent to fit a model with more than 1 predictor variable (and an intercept). Use the following parameters:
- model features = ‘sqft_living’, ‘sqft_living15’ (sqft_living15 is the average square feet of the nearest 15 neighbouring houses)
- output = ‘price’
- initial weights = [-100000, 1, 1] (intercept, sqft_living, and sqft_living_15 respectively)
- step size = 4e-12
- tolerance = 1e9

In [24]:
feature_matrix, output_array = get_numpy_data(train_data, ['sqft_living', 'sqft_living15'], "price")

multi_weights = gradient_descent_vectorized(feature_matrix, output_array, [-100000., 1., 1.], 4e-12, 1e9)

Use the regression weights from this second model (using sqft_living and sqft_living15) and predict the outcome of all the house prices on the TEST data.

In [26]:
test_multi_feature_matrix, test_output = get_numpy_data(test_data, ['sqft_living', 'sqft_living15'], "price")
new_predicted_house_prices = test_multi_feature_matrix @ multi_weights

**Quiz Question**: What is the predicted price for the 1st house in the TEST data set for model 2 (round to nearest dollar)?

In [29]:
first_house_price_multi_model = new_predicted_house_prices[0].round(0)
print ("Answer:", first_house_price_multi_model)

Answer: 366651.0


**Quiz Question**: Which estimate was closer to the true price for the 1st house on the TEST data set, model 1 or model 2?

In [36]:
first_house_actual_price = test_output[0]
diff_model_1 = abs(first_house_actual_price - first_house_price_simple_model)
diff_model_2 = abs(first_house_actual_price - first_house_price_multi_model)
print ("Model 1 estimate diff:", diff_model_1, "\nModel 2 estimate diff:", diff_model_2)

Model 1 estimate diff: 46134.0 
Model 2 estimate diff: 56651.0


Now compute RSS on all test data for the second model. Record the value and store it for later.

In [33]:
multi_RSS = ((new_predicted_house_prices - test_output) ** 2).sum()

**Quiz Question**: Which model (1 or 2) has lowest RSS on all of the TEST data?

In [35]:
print ("Model 1 RSS:", simple_RSS, "\nModel 2 RSS:", multi_RSS)

Model 1 RSS: 275400044902128.3 
Model 2 RSS: 270263443629803.56
